In [66]:
import os, glob
import sys
import settings
import json
import concurrent.futures
from google.cloud import pubsub_v1
import google.auth
import subprocess as sp
import time
import utils
import base64

In [48]:
list_res = utils.service.users().messages().list(userId='me', q='in:inbox', maxResults=1).execute()
messages = list_res.get('messages', [])
eid = messages[0]['id']
msg = utils.service.users().messages().get(
    userId='me', id=eid, format='full'
).execute()

email_from = ','.join([x['value'] for x in msg['payload']['headers'] if x['name'] == 'From'])

if 'info@account.netflix.com' not in email_from:
    # return
    exit(1)

In [52]:
import re

In [67]:
def get_plain_text_body(payload):
    """
    Recursively extracts and decodes the plain text body from a Gmail API payload.
    """
    # Case 1: Simple, single-part email
    if 'parts' not in payload:
        data = payload.get('body', {}).get('data', '')
        if data:
            return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
        return ''

    # Case 2: Multipart email (search recursively for 'text/plain')
    def extract_text(parts):
        for part in parts:
            if part.get('mimeType') == 'text/plain':
                data = part.get('body', {}).get('data', '')
                if data:
                    return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
            # Check nested sub-parts if present
            if 'parts' in part:
                result = extract_text(part['parts'])
                if result:
                    return result
        return ''

    return extract_text(payload['parts'])

In [70]:
payload = get_plain_text_body(msg['payload'])

In [83]:
payload

"Your Netflix temporary access code\r\n\r\nYour temporary access code\r\n\r\nHi Gino,\r\n\r\nWe received a request for a temporary access code from the\r\ndevice below.\r\n\r\nIf this was you, or someone who lives with you, you can get\r\na temporary access code to watch.\r\n\r\nThis code is for travel or temporary access outside your\r\nNetflix Household. Please don’t send this code to anyone\r\nelse.\r\n\r\n\xa0\r\n\r\n\xa0\r\n\xa0\r\n\r\nGet Code\r\n[https://www.netflix.com/account/travel/verify?nftoken=Bgj8vOvcAxK5AcbAqB/ZFWy0aOAWf0KknI7z2N7Xt52O5FNZtJPWhrEHNTKCoYk707QyBaADucEDvsJcKZr/56sjI/3NFhPzEqXFcvgTnV9jpIk2fvK4u6TCFcPrYOAkMFzrSl+80I44BHuP1Rgn0SiS43vDPzeC5/ktYwI6hA6vHv1ccShSXJx4H2HviqOz7Rm8jMWus8I4x/wLh3yoTcc8RDIVdPHmS5pXBiIlm0bd3aGWJL2+t7R98CQMvw71tZiFGAYiDgoMq9Ex2DyQ4knZjpQu&messageGuid=8e31d0a5-d3ef-4ce2-bee6-b4a064eef1f8]\r\n\r\n* Link expires after 15 minutes.\r\n\r\n\xa0\r\n\r\nKeep your account secure: If you don't know who this was, we\r\nrecommend that you immediately

In [89]:
re.findall('.*?https://.*?(?=[ \\]])', payload)

['[https://www.netflix.com/account/travel/verify?nftoken=Bgj8vOvcAxK5AcbAqB/ZFWy0aOAWf0KknI7z2N7Xt52O5FNZtJPWhrEHNTKCoYk707QyBaADucEDvsJcKZr/56sjI/3NFhPzEqXFcvgTnV9jpIk2fvK4u6TCFcPrYOAkMFzrSl+80I44BHuP1Rgn0SiS43vDPzeC5/ktYwI6hA6vHv1ccShSXJx4H2HviqOz7Rm8jMWus8I4x/wLh3yoTcc8RDIVdPHmS5pXBiIlm0bd3aGWJL2+t7R98CQMvw71tZiFGAYiDgoMq9Ex2DyQ4knZjpQu&messageGuid=8e31d0a5-d3ef-4ce2-bee6-b4a064eef1f8',
 '[https://www.netflix.com/ManageAccountAccess?g=8e31d0a5-d3ef-4ce2-bee6-b4a064eef1f8&lkid=URL_MANAGE_ACCOUNT_ACCESS&lnktrk=EVO&nftoken=Bgj8vOvcAxK5AaucKB0tyctfRzFMC%2BnnPb2zht4i6gK3jLK7R2TWKqzDpf5GUEJ%2F%2FfxVVUv19kEyolBkm0N8LtErprXxWjuOjd8sLaQ7MzO5UOSQsvN8%2FIaxGNPII6TSnP0UinUN%2BDGB5zdaBmu5MOmte6c1p5bdeGNWuw1FfPyTS61ZFTXKD8o0Eyk3HPzKuiFYVs%2BtsPrX9j1sD27ISYO607kgRaUp6fmDe9DWQ60gEsTo35G8vJ%2BXwhmONKYWWSyIGAYiDgoMj3WZSjfHKHIRI4QI',
 '[https://www.netflix.com/password?g=8e31d0a5-d3ef-4ce2-bee6-b4a064eef1f8&lkid=URL_PASSWORD&lnktrk=EVO&nftoken=Bgj8vOvcAxK4AXkTo6MpjOhqMsxmLKadwx2efne%2F9tjRO6LGad7CMVpo

In [73]:
payload

"Your Netflix temporary access code\r\n\r\nYour temporary access code\r\n\r\nHi Gino,\r\n\r\nWe received a request for a temporary access code from the\r\ndevice below.\r\n\r\nIf this was you, or someone who lives with you, you can get\r\na temporary access code to watch.\r\n\r\nThis code is for travel or temporary access outside your\r\nNetflix Household. Please don’t send this code to anyone\r\nelse.\r\n\r\n\xa0\r\n\r\n\xa0\r\n\xa0\r\n\r\nGet Code\r\n[https://www.netflix.com/account/travel/verify?nftoken=Bgj8vOvcAxK5AcbAqB/ZFWy0aOAWf0KknI7z2N7Xt52O5FNZtJPWhrEHNTKCoYk707QyBaADucEDvsJcKZr/56sjI/3NFhPzEqXFcvgTnV9jpIk2fvK4u6TCFcPrYOAkMFzrSl+80I44BHuP1Rgn0SiS43vDPzeC5/ktYwI6hA6vHv1ccShSXJx4H2HviqOz7Rm8jMWus8I4x/wLh3yoTcc8RDIVdPHmS5pXBiIlm0bd3aGWJL2+t7R98CQMvw71tZiFGAYiDgoMq9Ex2DyQ4knZjpQu&messageGuid=8e31d0a5-d3ef-4ce2-bee6-b4a064eef1f8]\r\n\r\n* Link expires after 15 minutes.\r\n\r\n\xa0\r\n\r\nKeep your account secure: If you don't know who this was, we\r\nrecommend that you immediately

"Your Netflix temporary access code\r\n\r\nYour temporary access code\r\n\r\nHi Gino,\r\n\r\nWe received a request for a temporary access code from the\r\ndevice below.\r\n\r\nIf this was you, or someone who lives with you, you can get\r\na temporary access code to watch.\r\n\r\nThis code is for travel or temporary access outside your\r\nNetflix Household. Please don’t send this code to anyone\r\nelse.\r\n\r\n\xa0\r\n\r\n\xa0\r\n\xa0\r\n\r\nGet Code\r\n[https://www.netflix.com/account/travel/verify?nftoken=Bgj8vOvcAxK5AcbAqB/ZFWy0aOAWf0KknI7z2N7Xt52O5FNZtJPWhrEHNTKCoYk707QyBaADucEDvsJcKZr/56sjI/3NFhPzEqXFcvgTnV9jpIk2fvK4u6TCFcPrYOAkMFzrSl+80I44BHuP1Rgn0SiS43vDPzeC5/ktYwI6hA6vHv1ccShSXJx4H2HviqOz7Rm8jMWus8I4x/wLh3yoTcc8RDIVdPHmS5pXBiIlm0bd3aGWJL2+t7R98CQMvw71tZiFGAYiDgoMq9Ex2DyQ4knZjpQu&messageGuid=8e31d0a5-d3ef-4ce2-bee6-b4a064eef1f8]\r\n\r\n* Link expires after 15 minutes.\r\n\r\n\xa0\r\n\r\nKeep your account secure: If you don't know who this was, we\r\nrecommend that you immediately